In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb

# Rutas de los ficheros que necesitamos
RUTA_MODELO  = "Modelos/modelo_lgb_optuna.txt"
RUTA_DATASET = "df_entrenamiento.parquet"

# Las 25 features EXACTAS con las que se entrenó el modelo (en el mismo orden)
FEATURES = [
    "hora_dia", "dia_semana", "mes", "es_finde", "es_festivo",
    "ocupation_lag_24h", "ocupation_lag_168h", "ocupation_media_7d",
    "salidas_lag_24h", "salidas_lag_168h", "salidas_media_24h",
    "entradas_lag_24h", "entradas_lag_168h", "entradas_media_24h",
    "temperatura", "precipitacion", "viento",
    "oficinas", "restaurantes_bares", "comercio", "educacion",
    "transporte", "deporte", "ocio",
    "total_bases",
]

In [3]:
# Cargar el modelo entrenado desde disco
modelo = lgb.Booster(model_file=RUTA_MODELO)
print(f"✓ Modelo cargado: {RUTA_MODELO}")
print(f"  Features que espera el modelo: {modelo.num_feature()}")

# Cargar el dataset completo
df = pd.read_parquet(RUTA_DATASET)
print(f"\n✓ Dataset cargado: {len(df):,} filas")

# Nos quedamos solo con el periodo de validación (oct-dic 2022),
# que es sobre el que la app mostrará predicciones
df_val = df[df["hora"] >= pd.Timestamp("2022-10-01")].copy()
print(f"  Periodo de validación: {len(df_val):,} filas")
print(f"  Rango: {df_val['hora'].min()} → {df_val['hora'].max()}")

✓ Modelo cargado: Modelos/modelo_lgb_optuna.txt
  Features que espera el modelo: 25

✓ Dataset cargado: 2,220,323 filas
  Periodo de validación: 550,974 filas
  Rango: 2022-10-01 01:00:00 → 2022-12-31 23:00:00


In [4]:
# Preparar las features (filtrar filas sin target ni features completas no hace falta:
# LightGBM admite NaN, pero quitamos filas sin ocupación real para poder comparar)
X = df_val[FEATURES]

# Predecir: el modelo devuelve probabilidades de las 3 clases
proba = modelo.predict(X)
categoria_pred = proba.argmax(axis=1)

print(f"✓ Predicciones generadas para {len(categoria_pred):,} filas")
print(f"\nDistribución de las predicciones:")
nombres_cat = {0: "BAJA", 1: "NORMAL", 2: "ALTA"}
for cat, n in zip(*np.unique(categoria_pred, return_counts=True)):
    print(f"  {nombres_cat[cat]:8s}: {n:,} ({n/len(categoria_pred)*100:.1f}%)")

✓ Predicciones generadas para 550,974 filas

Distribución de las predicciones:
  BAJA    : 189,815 (34.5%)
  NORMAL  : 300,666 (54.6%)
  ALTA    : 60,493 (11.0%)


In [5]:
df_app = df_val.copy()
df_app["pred_categoria"] = categoria_pred
df_app["prob_baja"]   = proba[:, 0]
df_app["prob_normal"] = proba[:, 1]
df_app["prob_alta"]   = proba[:, 2]

# Solo las columnas que la app necesita
columnas_app = [
    "id", "name", "latitude", "longitude", "total_bases",
    "hora", "hora_dia", "dia_semana",
    "ocupation",
    "pred_categoria", "prob_baja", "prob_normal", "prob_alta",
]
df_app = df_app[columnas_app]

df_app.to_parquet("predicciones_app.parquet", index=False)

print(f"✓ Guardado: predicciones_app.parquet")
print(f"  Filas: {len(df_app):,}")
print(f"  Columnas: {list(df_app.columns)}")
print(f"\nMuestra:")
print(df_app.head())

✓ Guardado: predicciones_app.parquet
  Filas: 550,974
  Columnas: ['id', 'name', 'latitude', 'longitude', 'total_bases', 'hora', 'hora_dia', 'dia_semana', 'ocupation', 'pred_categoria', 'prob_baja', 'prob_normal', 'prob_alta']

Muestra:
      id              name   latitude  longitude  total_bases  \
6320   1  Puerta del Sol A  40.417214  -3.701834           30   
6321   1  Puerta del Sol A  40.417214  -3.701834           30   
6322   1  Puerta del Sol A  40.417214  -3.701834           30   
6323   1  Puerta del Sol A  40.417214  -3.701834           30   
6324   1  Puerta del Sol A  40.417214  -3.701834           30   

                    hora  hora_dia  dia_semana  ocupation  pred_categoria  \
6320 2022-10-01 01:00:00         1           5   0.555556               0   
6321 2022-10-01 02:00:00         2           5   0.444444               0   
6322 2022-10-01 03:00:00         3           5   0.333333               0   
6323 2022-10-01 04:00:00         4           5   0.185185       